# Lista 8
### Maria Nowacka 275981

In [ ]:
!pip install tensorflow

In [ ]:
!pip install kagglehub

In [ ]:
!pip install nltk

In [55]:
import pandas as pd, os, numpy as np, re, string, nltk
from tensorflow.keras.datasets import imdb
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### zadanie 1
Wczytanie i eksploracja danych: Wczytaj publicznie dostępny zbiór
danych do analizy wydźwięku (np. zbiór recenzji filmowych z IMDB).
Zapoznaj się z danymi: sprawdź liczbę recenzji, rozkład klas, i długość
tekstu.

In [27]:
data = pd.read_csv("IMDBDataset.csv")

In [29]:
total_reviews = len(data)
print(f"Liczba recenzji: {total_reviews}")

Liczba recenzji: 50000


In [31]:
class_distribution = data['sentiment'].value_counts()
print("\nRozkład klas:", class_distribution)


Rozkład klas: sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [33]:
data['review_length'] = data['review'].apply(len)

print("\nDługość tekstów:")
print(data['review_length'].describe())


Długość tekstów:
count    50000.000000
mean      1309.431020
std        989.728014
min         32.000000
25%        699.000000
50%        970.000000
75%       1590.250000
max      13704.000000
Name: review_length, dtype: float64


### zadanie 2
Przygotowanie danych tekstowych: Przeprowadź pre-processing
danych tekstowych, który powinien obejmować co najmniej:
* usunięcie znaków interpunkcyjnych i cyfr,
* konwersję tekstu na małe litery,
* usunięcie stopwords (słów-wypełniaczy).

In [39]:
data['review'][0] # przykładowa "surowa" recenzja

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [50]:
nltk.download('stopwords', quiet=True)
english_stopwords = set(stopwords.words('english'))

In [48]:
def preprocess_text(text):
    text = text.lower()

    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    words = text.split()
    words = [word for word in words if word not in english_stopwords]
    
    return ' '.join(words)

data['processed_review'] = data['review'].apply(preprocess_text)

data.loc[0]['processed_review']

'one reviewers mentioned watching oz episode youll hooked right exactly happened mebr br first thing struck oz brutality unflinching scenes violence set right word go trust show faint hearted timid show pulls punches regards drugs sex violence hardcore classic use wordbr br called oz nickname given oswald maximum security state penitentary focuses mainly emerald city experimental section prison cells glass fronts face inwards privacy high agenda em city home manyaryans muslims gangstas latinos christians italians irish moreso scuffles death stares dodgy dealings shady agreements never far awaybr br would say main appeal show due fact goes shows wouldnt dare forget pretty pictures painted mainstream audiences forget charm forget romanceoz doesnt mess around first episode ever saw struck nasty surreal couldnt say ready watched developed taste oz got accustomed high levels graphic violence violence injustice crooked guards wholl sold nickel inmates wholl kill order get away well mannered 

### zadanie 3
Wektoryzacja tekstu: Przekształć dane tekstowe na format numeryczny, który może być użyty przez model uczenia maszynowego. Zastosuj jedną z następujących technik:
* Bag-of-Words (Bag of words): Użyj `CountVectorizer` lub
`TfidfVectorizer` z scikit-learn.
* wektoryzacja słów (Word Embeddings): Użyj gotowych wytrenowanych wektorów (np. `GloVe`) lub wytrenuj własne, jeśli czas
na to pozwala.

In [52]:
X = data['processed_review']
y = data['sentiment'].apply(lambda x: 1 if x == 'positive' else 0) 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y
)

print(f"Liczba próbek w zbiorze treningowym: {len(X_train)}")
print(f"Liczba próbek w zbiorze testowym: {len(X_test)}")

Liczba próbek w zbiorze treningowym: 40000
Liczba próbek w zbiorze testowym: 10000


In [53]:
tfidf_vectorizer = TfidfVectorizer(
    min_df=5, 
    max_features=10000
)
tfidf_vectorizer.fit(X_train)
print(f"Rozmiar wygenerowanego słownika (liczba cech): {len(tfidf_vectorizer.vocabulary_)}")

Rozmiar wygenerowanego słownika (liczba cech): 10000


In [54]:
X_train_tfidf = tfidf_vectorizer.transform(X_train)

X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("\n--- Wyniki Wektoryzacji ---")
print(f"Wektoryzacja X_train (próbki, cechy): {X_train_tfidf.shape}")
print(f"Wektoryzacja X_test (próbki, cechy): {X_test_tfidf.shape}")


--- Wyniki Wektoryzacji ---
Wektoryzacja X_train (próbki, cechy): (40000, 10000)
Wektoryzacja X_test (próbki, cechy): (10000, 10000)


### zadanie 4
Budowa modelu klasyfikacji: Wytrenuj klasyfikator na przygotowanych danych. Możesz użyć:
* Klasycznego modelu ML (LogisticRegression, Naive Bayes)
lub,
* Prostej sieci neuronowej z listy 7 (dla wektoryzacji Bag-of-Words).

In [59]:
model = LogisticRegression(solver='liblinear')
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

### zadanie 5
Ocena i interpretacja wyników: Oceń wydajność wytrenowanego
modelu na zbiorze testowym, używając metryk takich jak dokładność,
precyzja i czułość. Dodatkowo, przeanalizuj macierz pomyłek.

In [60]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Dokładność (Accuracy) na zbiorze testowym: {accuracy:.4f}")

print("Raport Klasyfikacji:\n")
print(classification_report(y_test, y_pred, target_names=['Negatywne (0)', 'Pozytywne (1)']))

conf_matrix = confusion_matrix(y_test, y_pred)
print("Macierz Pomyłek (Confusion Matrix):\n")
print(conf_matrix)

Dokładność (Accuracy) na zbiorze testowym: 0.8863
Raport Klasyfikacji:

               precision    recall  f1-score   support

Negatywne (0)       0.89      0.88      0.89      5000
Pozytywne (1)       0.88      0.90      0.89      5000

     accuracy                           0.89     10000
    macro avg       0.89      0.89      0.89     10000
 weighted avg       0.89      0.89      0.89     10000

Macierz Pomyłek (Confusion Matrix):

[[4386  614]
 [ 523 4477]]


* Dokładność na poziomie 88-89% dla tak prostego modelu jest naprawdę okay. \
* Precyzja [0]: 89% recenzji, które model oznaczył jako negatywne, faktycznie były negatywne. (Niski wskaźnik fałszywie negatywnych)
* Czułość [0]: 88% wszystkich faktycznie negatywnych recenzji zostało poprawnie zidentyfikowanych przez model. \
* Precyzja [1]:88% recenzji, które model oznaczył jako pozytywne, faktycznie było pozytywnych. 
* Czułość [1]: 90% wszystkich faktycznie pozytywnych recenzji zostało poprawnie zidentyfikowanych przez model. (Wysoka zdolność do znajdowania pozytywów)
* Macierz pomyłek: \
  4386 (TN) 614 (FP) \
523 (FN) 4477 (TP) 

### zadanie 6

In [100]:
new_reviews = [
    "Best movie I've seen this year! Had lot of fun and giggles",  # pozytywna
    "Such a waste of time. Better to watch your laundry air-dry.",  # negatywna
    "It was okay but didn't steal my heart", # neutralna
    "Had good time for like 5 minutes, the rest was not so good", # neutralna/negatywna
    "Not bad for chilling with friends", # neutralne/pozytywna
    "The beginning didn't set my expectations too high but the plot unraveled an intruiging story. Would watch again"
]

new_data = pd.DataFrame({'review': new_reviews, 'expected_sentiment': [1,0,0,0,1,1]})

In [101]:
new_data['processed_review'] = new_data['review'].apply(preprocess_text)
X_new_tfidf = tfidf_vectorizer.transform(new_data['processed_review'])
predictions = model.predict(X_new_tfidf)
new_data['predicted_sentiment'] = predictions

In [102]:
new_data

,review,expected_sentiment,processed_review,predicted_sentiment
0,Best movie I've seen this year! Had lot of fun...,1,best movie ive seen year lot fun giggles,1
1,Such a waste of time. Better to watch your lau...,0,waste time better watch laundry airdry,0
2,It was okay but didn't steal my heart,0,okay didnt steal heart,0
3,"Had good time for like 5 minutes, the rest was...",0,good time like minutes rest good,0
4,Not bad for chilling with friends,1,bad chilling friends,0
5,The beginning didn't set my expectations too h...,1,beginning didnt set expectations high plot unr...,0


Model poprawnie zaklasyfikował mocno pozytywną oraz mocno negatywną opinię. Dla opini neutralnych lub lekko negatywnych przypisał klasę "negatywna" (0). Dla dwóch ostatnich opini model również przypisał "0", mimo lekko pozytywnego (w zamierzeniu) wydźwięku.